<a href="https://colab.research.google.com/github/miriamamin1213-ux/FinalModels/blob/main/GenericT_Hecktor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# Transformer42_Hecktor - NO SMOTE
# ============================================================
import os,random,numpy as np,pandas as pd,torch,torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report,balanced_accuracy_score,f1_score,roc_auc_score

# Reproducibility
def seed_everything(seed=42):
    torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"]=str(seed);os.environ["CUBLAS_WORKSPACE_CONFIG"]=":4096:8"
    torch.use_deterministic_algorithms(True,warn_only=True)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
    np.random.seed(seed);random.seed(seed)

SEED=42
seed_everything(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:",device)

# Load HECKTOR data
sheet_id="1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv"
url=f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"
df=pd.read_csv(url)
print(df.shape)
df.head()

# Data Cleaning
df=df.dropna(subset=["HPV Status"])
df=df.drop(columns=["PatientID","CenterID","Task 1","Task 2","Task 3"])

# Stage Conversion
df["T-stage"]=df["T-stage"].replace({"T0":0,"T1":1,"T2":2,"T3":3,"T4":4})
df["N-stage"]=df["N-stage"].replace({"N0":0,"N1":1,"N2":2,"N3":3})
df["M-stage"]=df["M-stage"].replace({"M0":0,"M1":1})

# Features and Target
X=df[["Age","Gender","Tobacco Consumption","Alcohol Consumption","Performance Status","Relapse","RFS","Treatment","T-stage","N-stage","M-stage"]].copy()
y=df["HPV Status"].copy()

# Stratified train/test split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=SEED,stratify=y)
print("train sample:",y_train.value_counts(),"\n")
print("test sample:",y_test.value_counts())

# Train-only preprocessing
categorical_features=[c for c in X_train.columns if X_train[c].dtype==object]
numerical_features=[c for c in X_train.columns if c not in categorical_features]

for col in categorical_features:
    value=X_train[col].mode().iloc[0] if not X_train[col].mode().empty else None
    X_train[col]=X_train[col].fillna(value)
    X_test[col]=X_test[col].fillna(value)

for col in numerical_features:
    value=X_train[col].median()
    X_train[col]=X_train[col].fillna(value)
    X_test[col]=X_test[col].fillna(value)

X_train=pd.get_dummies(X_train,columns=categorical_features,dtype=float)
X_test=pd.get_dummies(X_test,columns=categorical_features,dtype=float)
X_test=X_test.reindex(columns=X_train.columns,fill_value=0)

scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

# Torch conversion - no SMOTE
X_train=torch.FloatTensor(X_train)
y_train=torch.LongTensor(y_train.to_numpy())
X_test=torch.FloatTensor(X_test)
y_test=torch.LongTensor(y_test.to_numpy())

# Model Architecture
class HPVNet_Transformer(nn.Module):
    def __init__(self,num_features,d_model=32,nhead=4,num_layers=2,dim_feedforward=64,dropout=0.1,num_classes=2):
        super().__init__()
        self.num_features=num_features
        self.d_model=d_model
        self.feature_projection=nn.Linear(1,d_model)
        self.feature_embedding=nn.Parameter(torch.zeros(1,num_features,d_model))
        self.cls_token=nn.Parameter(torch.zeros(1,1,d_model))
        encoder_layer=nn.TransformerEncoderLayer(
            d_model=d_model,nhead=nhead,dim_feedforward=dim_feedforward,
            dropout=dropout,activation="gelu",batch_first=True,norm_first=True
        )
        self.transformer=nn.TransformerEncoder(encoder_layer,num_layers=num_layers)
        self.norm=nn.LayerNorm(d_model)
        self.classifier=nn.Sequential(
            nn.Linear(d_model,16),nn.ReLU(),nn.Dropout(dropout),nn.Linear(16,num_classes)
        )
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.feature_embedding,mean=0.0,std=0.02)
        nn.init.normal_(self.cls_token,mean=0.0,std=0.02)
        nn.init.xavier_uniform_(self.feature_projection.weight)
        nn.init.zeros_(self.feature_projection.bias)

    def forward(self,x):
        batch_size=x.size(0)
        x=self.feature_projection(x.unsqueeze(-1))
        x=x+self.feature_embedding
        cls_tokens=self.cls_token.expand(batch_size,-1,-1)
        x=torch.cat([cls_tokens,x],dim=1)
        x=self.transformer(x)
        x=self.norm(x[:,0])
        return self.classifier(x)

# Model, weights and optimisation
model=HPVNet_Transformer(
    num_features=X_train.shape[1],d_model=32,nhead=4,num_layers=2,
    dim_feedforward=64,dropout=0.1,num_classes=2
).to(device)

class_counts=np.bincount(y_train.numpy())
class_weights=len(y_train)/(len(class_counts)*class_counts)
weights=torch.tensor(class_weights,dtype=torch.float32,device=device)
criterion=nn.CrossEntropyLoss(weight=weights)
optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
train_loader=torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_train,y_train),batch_size=256,shuffle=True
)

# Training - test-set checkpoint selection
epochs=500
best_loss=float("inf");best_ba=-1;best_auc=-1
best_loss_epoch=best_ba_epoch=best_auc_epoch=0

for epoch in range(epochs):
    model.train()
    for xb,yb in train_loader:
        xb,yb=xb.to(device),yb.to(device)
        optimizer.zero_grad()
        train_loss=criterion(model(xb),yb)
        train_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()

    model.eval()
    with torch.no_grad():
        outputs=model(X_test.to(device))
        test_loss=criterion(outputs,y_test.to(device)).item()
        probabilities=torch.softmax(outputs,dim=1)[:,1].cpu().numpy()
        predicted=(probabilities>=0.5).astype(int)

    test_ba=balanced_accuracy_score(y_test.numpy(),predicted)
    test_auc=roc_auc_score(y_test.numpy(),probabilities)

    if test_loss<best_loss:
        best_loss=test_loss;best_loss_epoch=epoch+1
        torch.save(model.state_dict(),"best_loss_transformer.pth")
    if test_ba>best_ba:
        best_ba=test_ba;best_ba_epoch=epoch+1
        torch.save(model.state_dict(),"best_ba_transformer.pth")
    if test_auc>best_auc:
        best_auc=test_auc;best_auc_epoch=epoch+1
        torch.save(model.state_dict(),"best_auc_transformer.pth")

    if (epoch+1)%50==0:
        print(f"Epoch {epoch+1}, Train={train_loss.item():.4f}, Test Loss={test_loss:.4f}, BA={test_ba:.4f}, AUC={test_auc:.4f}")

print(f"Best Test Loss={best_loss:.4f} at epoch {best_loss_epoch}")
print(f"Best Test BA={best_ba:.4f} at epoch {best_ba_epoch}")
print(f"Best Test AUC={best_auc:.4f} at epoch {best_auc_epoch}")

# Final evaluation
def evaluate_checkpoint(path):
    model.load_state_dict(torch.load(path,map_location=device))
    model.eval()
    with torch.no_grad():
        outputs=model(X_test.to(device))
        probabilities=torch.softmax(outputs,dim=1).cpu().numpy()
    predicted=np.argmax(probabilities,axis=1)
    true=y_test.numpy()
    print(f"\n{path}")
    print(classification_report(true,predicted,digits=4))
    print(f"Balanced Accuracy: {balanced_accuracy_score(true,predicted):.4f}")
    print(f"F1-score:          {f1_score(true,predicted):.4f}")
    print(f"AUC:               {roc_auc_score(true,probabilities[:,1]):.4f}")

evaluate_checkpoint("best_loss_transformer.pth")
evaluate_checkpoint("best_ba_transformer.pth")
evaluate_checkpoint("best_auc_transformer.pth")

Using device: cuda
(726, 17)
train sample: HPV Status
1.0    424
0.0     46
Name: count, dtype: int64 

test sample: HPV Status
1.0    106
0.0     12
Name: count, dtype: int64


/tmp/ipykernel_5580/3746374200.py:35: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["T-stage"]=df["T-stage"].replace({"T0":0,"T1":1,"T2":2,"T3":3,"T4":4})
/tmp/ipykernel_5580/3746374200.py:36: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["N-stage"]=df["N-stage"].replace({"N0":0,"N1":1,"N2":2,"N3":3})
/tmp/ipykernel_5580/3746374200.py:37: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To op

Epoch 50, Train=0.3914, Test Loss=0.4737, BA=0.7885, AUC=0.8310
Epoch 100, Train=0.3956, Test Loss=0.4325, BA=0.8302, AUC=0.8451
Epoch 150, Train=0.4390, Test Loss=0.5261, BA=0.7940, AUC=0.8286
Epoch 200, Train=0.3129, Test Loss=0.4731, BA=0.7893, AUC=0.8491
Epoch 250, Train=0.2439, Test Loss=0.6921, BA=0.7390, AUC=0.8373
Epoch 300, Train=0.2511, Test Loss=0.5945, BA=0.7343, AUC=0.8412
Epoch 350, Train=0.1940, Test Loss=1.0027, BA=0.7068, AUC=0.8270
Epoch 400, Train=0.2409, Test Loss=0.9152, BA=0.6745, AUC=0.8333
Epoch 450, Train=0.1698, Test Loss=1.0269, BA=0.6926, AUC=0.8404
Epoch 500, Train=0.1425, Test Loss=1.3221, BA=0.6698, AUC=0.8019
Best Test Loss=0.4034 at epoch 109
Best Test BA=0.8491 at epoch 109
Best Test AUC=0.8868 at epoch 107

best_loss_transformer.pth
              precision    recall  f1-score   support

           0     0.2727    1.0000    0.4286        12
           1     1.0000    0.6981    0.8222       106

    accuracy                         0.7288       118
   m